# Polymorphic fields

A field is **polymorphic** ("multiform") when its value is not always the same
JSON type across records -- a `quantity` that is `"35 nm"` (string) in one
record, `35` (number) in another, `{"value": 35, "unit": "nm"}` (object) in a
third.

JSON Schema lets you declare this with a list-valued `type`, and this evaluator
supports it directly:

- `{"type": ["string", "null"]}` -- nullable; the `null` is dropped, so it is
  just a `string` field.
- `{"type": ["string", "number", "object"]}` -- a genuine multi-shape field.

`anyOf` / `oneOf` / `if`-`then`-`else` in JSON Schema might also generate multiple types.


## The problem: the default comparator is coarse

Declare `quantity` honestly as `["string", "number", "object"]`, you need a custom comparator to deal with the different types.

In [ ]:
from struct_extract_eval import evaluate
from struct_extract_eval.core.comparators.comparator import ComparatorResult
from struct_extract_eval.core.comparators.registry import register as register_comparator
from struct_extract_eval.core.xeval import annotate_xeval


def show(title, result):
    print(title)
    for rec in result.records:
        for fr in rec.field_results:
            print(f"  [{rec.record_id}] {fr.path:<8} gold={fr.gold_value!r:<24} extracted={fr.extracted_value!r:<20} -> {fr.status}")
    print(f"  mean F1={result.mean_f1:.2f}")


GOLD = [
    {"quantity": {"value": 35, "unit": "nm"}},  # object form
    {"quantity": 50},                            # number form
    {"quantity": "60 nm"},                       # string form
]
EXTRACTED = [
    {"quantity": "35 nm"},   # same meaning as gold, different shape
    {"quantity": "50"},      # same meaning as gold, different shape
    {"quantity": "65 nm"},   # genuinely wrong (60 != 65)
]

# Honest multi-type declaration. No comparator -> default `exact`.
# annotate_xeval assigns that default (evaluate does not annotate for you).
schema_default = {
    "type": "object",
    "properties": {"quantity": {"type": ["string", "number", "object"]}},
}
annotate_xeval(schema_default)

show("default exact comparator", evaluate(GOLD, EXTRACTED, schema_default))


In [ ]:
def _to_value_unit(v):
    if isinstance(v, dict):
        return v.get("value"), v.get("unit")
    if isinstance(v, (int, float)):
        return float(v), None
    if isinstance(v, str):
        parts = v.split()
        try:
            value = float(parts[0]) if parts else None
        except ValueError:
            # not a number (e.g. "n/a"): compare as raw text, never crash the run
            return v.strip(), None
        unit = parts[1] if len(parts) > 1 else None
        return value, unit
    return None, None


def compare_quantity(gold, extracted, params):
    if _to_value_unit(gold) == _to_value_unit(extracted):
        return ComparatorResult(score=1.0, comparator="quantity")
    return ComparatorResult(score=0.0, comparator="quantity", reason=f"{gold!r} != {extracted!r}")


register_comparator("quantity", compare_quantity, overwrite=True)

schema_poly = {
    "type": "object",
    "properties": {
        "quantity": {"type": ["string", "number", "object"], "x-eval-compare": "quantity"},
    },
}
annotate_xeval(schema_poly)   # no-op for `quantity` (it already has a comparator)

show("with quantity comparator", evaluate(GOLD, EXTRACTED, schema_poly))

Now the two cross-shape-but-equal records match, and the genuinely wrong one
(`60 nm` vs `65 nm`) is still a mismatch. One comparator, every shape covered.

## Validating polymorphic gold

By default `validate_gold` treats `type` as a hint (mismatches only warn). For a
multi-type field it accepts gold of any declared shape. If you want to *enforce*
that gold is one of the declared types, pass `strict_types=True`:

```python
from struct_extract_eval import validate_gold

validate_gold(GOLD, schema_poly)
validate_gold(GOLD, schema_poly, strict_types=True)  # raises GoldValidationError if a gold value is not one of string/number/object (null is exempt)
```